# 03_Model_Training - Random Forest (Improved Job Grouping)

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib
import matplotlib.pyplot as plt
import seaborn as sns
%matplotlib inline

In [ ]:
# Load the NEW cleaned dataset
df = pd.read_csv('ds_salaries_cleaned_v2.csv')
print('Job categories:\n', df['job_category'].value_counts())

In [ ]:
target = 'salary_in_usd'
X = df.drop(columns=[target])
y = df[target]

categorical_features = ['employment_type', 'job_category']
numeric_features = ['work_year', 'remote_ratio', 'experience_level_enc', 
                    'company_size_enc', 'is_us_company', 'is_us_residence']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, random_state=42)

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_features),
        ('num', 'passthrough', numeric_features)
    ]
)

model_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('regressor', RandomForestRegressor(random_state=42, n_jobs=-1))
])

param_grid = {
    'regressor__n_estimators': [300, 400, 500],
    'regressor__max_depth': [15, 18, 20, None],
    'regressor__min_samples_split': [2, 4, 6],
    'regressor__min_samples_leaf': [1, 2],
    'regressor__max_features': ['sqrt', 0.8, None]
}

grid_search = GridSearchCV(model_pipeline, param_grid, cv=5, scoring='neg_mean_absolute_error', n_jobs=-1, verbose=1)
grid_search.fit(X_train, y_train)

print("Best params:", grid_search.best_params_)

In [ ]:
best_model = grid_search.best_estimator_
y_test_pred = best_model.predict(X_test)

mae = mean_absolute_error(y_test, y_test_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_test_pred))
r2 = r2_score(y_test, y_test_pred)

print(f"Test MAE : ${mae:,.0f}")
print(f"Test RMSE: ${rmse:,.0f}")
print(f"Test R²  : {r2:.4f}")

In [ ]:
joblib.dump(best_model, 'random_forest_model_improved.joblib')
print("Model saved as random_forest_model_improved.joblib")